# Ved — Training on Kaggle

**Before running:**
1. Enable GPU: *Session options → Accelerator → GPU T4 × 1* (or P100)
2. Add a GitHub token as a Kaggle Secret named `GITHUB_TOKEN`  
   *(Kaggle → top-right menu → Secrets → Add Secret)*

**What this notebook does:**
- Streams 2B tokens from FineWeb-Edu (~4 GB on disk — fits on Kaggle)
- Trains Ved (~50M params, ~4-5 hrs per session)
- Saves the checkpoint to `/kaggle/working/` (download from Output tab)
- Pushes training logs + generated samples back to `orcus108/ved`

**Resuming across sessions:** set `INIT_FROM = 'resume'` in cell 4,  
then attach your saved checkpoint dataset (see instructions at the bottom).

In [ ]:
# ── 1. Install deps ──────────────────────────────────────────────────────────
!pip install -q tiktoken datasets tqdm

In [ ]:
# ── 2. Clone repo + configure git with GitHub token ──────────────────────────
import os
from kaggle_secrets import UserSecretsClient

REPO_URL = "https://github.com/orcus108/ved.git"

secrets      = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")

AUTH_URL = REPO_URL.replace("https://", f"https://orcus108:{GITHUB_TOKEN}@")

!git clone {REPO_URL} ved
%cd ved
!git remote set-url origin {AUTH_URL}
!git config user.email "ved-kaggle@train"
!git config user.name  "Ved Kaggle"

In [ ]:
# ── 3. Verify GPU ────────────────────────────────────────────────────────────
import torch
print(f"CUDA : {torch.cuda.is_available()}")
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"PyTorch {torch.__version__}")

In [ ]:
# ── 4. Config — edit here ────────────────────────────────────────────────────
INIT_FROM  = 'scratch'     # 'scratch' for first session, 'resume' after
OUT_DIR    = 'out-ved-50m'
MAX_ITERS  = 7600          # 2B tokens ÷ 262K tokens/step ≈ 4-5 hrs on T4/P100
WANDB_LOG  = False         # set True and add WANDB_API_KEY secret to enable

In [ ]:
# ── 5. (Resume only) Restore checkpoint from Kaggle dataset ──────────────────
# If INIT_FROM='resume', mount your checkpoint dataset in the notebook
# (right sidebar → Add data → Your datasets → ved-checkpoints)
# then run this cell to copy it into place.

if INIT_FROM == 'resume':
    import shutil, glob
    ckpt_candidates = glob.glob("/kaggle/input/**/ckpt.pt", recursive=True)
    assert ckpt_candidates, "No ckpt.pt found — attach your checkpoint dataset first."
    os.makedirs(OUT_DIR, exist_ok=True)
    shutil.copy(ckpt_candidates[0], f"{OUT_DIR}/ckpt.pt")
    print(f"Restored checkpoint from {ckpt_candidates[0]}")

In [ ]:
# ── 6. Stream + tokenise FineWeb-Edu (2B tokens, ~4 GB) ──────────────────────
# Uses streaming mode — never downloads the full 25 GB corpus.
# Takes ~20-40 min. Skipped on resume if data already exists.

TRAIN_BIN = "data/fineweb_edu/train.bin"

if not os.path.exists(TRAIN_BIN):
    print("Preparing data (streaming 2B tokens) …")
    !python data/fineweb_edu/prepare.py
else:
    size_gb = os.path.getsize(TRAIN_BIN) / 1e9
    print(f"Data ready ({size_gb:.1f} GB).")

In [ ]:
# ── 7. Train ─────────────────────────────────────────────────────────────────
wandb_flag = '' if WANDB_LOG else '--no_wandb'

!python train.py config/train_ved.py \
    --init_from={INIT_FROM} \
    --out_dir={OUT_DIR}     \
    --max_iters={MAX_ITERS} \
    {wandb_flag}            \
    2>&1 | tee train_log.txt

In [ ]:
# ── 8. Generate samples ──────────────────────────────────────────────────────
CKPT   = f"{OUT_DIR}/ckpt.pt"
PROMPTS = [
    "The universe began",
    "Scientists recently discovered",
    "Once upon a time",
]

with open("samples.txt", "w") as f:
    for prompt in PROMPTS:
        result = !python sample.py \
            --ckpt={CKPT}          \
            --prompt="{prompt}"    \
            --max_new_tokens=200   \
            --temperature=0.8      \
            --top_k=50             \
            --num_samples=1        \
            --device=cuda
        block = "\n".join(result)
        f.write(f"=== {prompt} ===\n{block}\n\n")
        print(f"\n=== {prompt} ===\n{block}")

In [ ]:
# ── 9. Save checkpoint to Kaggle output ──────────────────────────────────────
import shutil

ckpt_src = f"{OUT_DIR}/ckpt.pt"
ckpt_dst = "/kaggle/working/ckpt.pt"

if os.path.exists(ckpt_src):
    shutil.copy(ckpt_src, ckpt_dst)
    size_mb = os.path.getsize(ckpt_dst) / 1e6
    print(f"Checkpoint saved: {ckpt_dst}  ({size_mb:.0f} MB)")
    print("Download from the Output tab → upload as a Kaggle Dataset to resume.")
else:
    print("No checkpoint found.")

In [ ]:
# ── 10. Push logs + samples to GitHub ────────────────────────────────────────
# Checkpoint is NOT pushed (too large for GitHub).
# train_log.txt + samples.txt + config go to a results/<run-id> branch.

import subprocess, time

RUN_ID = f"run-{int(time.time())}"
BRANCH = f"results/{RUN_ID}"

try:
    ckpt   = torch.load(ckpt_src, map_location="cpu")
    n_iter = ckpt.get("iter_num", "?")
    loss   = f"{ckpt.get('best_val_loss', 0):.4f}"
except Exception:
    n_iter, loss = "?", "?"

os.makedirs("results", exist_ok=True)
shutil.copy("train_log.txt",       f"results/{RUN_ID}_train_log.txt")
shutil.copy("samples.txt",         f"results/{RUN_ID}_samples.txt")
shutil.copy("config/train_ved.py", f"results/{RUN_ID}_config.py")

cmds = [
    ["git", "checkout", "-b", BRANCH],
    ["git", "add",
     f"results/{RUN_ID}_train_log.txt",
     f"results/{RUN_ID}_samples.txt",
     f"results/{RUN_ID}_config.py"],
    ["git", "commit", "-m",
     f"results: {RUN_ID} | iters={n_iter} | val_loss={loss}"],
    ["git", "push", "origin", BRANCH],
]

for cmd in cmds:
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout or f"OK: {' '.join(cmd)}")
    if r.returncode != 0:
        print(f"STDERR: {r.stderr}")

print(f"\nResults pushed to: https://github.com/orcus108/ved/tree/{BRANCH}")

## Resuming in a new session

1. **Download** `ckpt.pt` from the Output tab
2. **Upload** it as a new Kaggle Dataset (name it `ved-checkpoints`)
3. In a new notebook: **Add data** → your `ved-checkpoints` dataset
4. Set `INIT_FROM = 'resume'` and increase `MAX_ITERS` in cell 4
5. Run all cells — cell 5 auto-copies the checkpoint into place

## Enabling wandb

1. Add your W&B API key as a Kaggle Secret named `WANDB_API_KEY`
2. In cell 1 add: `!pip install -q wandb`
3. In cell 2 add (after secrets setup):
   ```python
   import wandb
   wandb.login(key=secrets.get_secret('WANDB_API_KEY'))
   ```
4. Set `WANDB_LOG = True` in cell 4